In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
target_dir = '/content/drive/MyDrive/CS-GY6953/Proj1/proj1(augmix)'
os.chdir(target_dir)
print(f"Current working directory: {os.getcwd()}")

## Packages:

In [ ]:
# File paths, PyTorch model operations, and optimization utilities used throughout the notebook.
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

# Read image annotations and save per-epoch metrics as tables.
import pandas as pd
import time

# Open image files before applying tensor conversion and augmentation.
from PIL import Image

# Create a reproducible split that preserves class proportions.
from sklearn.model_selection import train_test_split

# Dataset defines individual examples; DataLoader groups them into batches.
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Inspect model dimensions and parameter counts with a sample input.
from torchinfo import summary
from pathlib import Path

# Choose a progress-bar display suitable for a notebook or terminal.
from tqdm.auto import tqdm

# Graphing and manual checking
import matplotlib.pyplot as plt
import numpy as np

## Data Loading:

### unzip data (google colab specific)

In [ ]:
!unzip -q "/content/drive/MyDrive/CS-GY6953/Proj1/data.zip" -d "/content/data"

In [ ]:
!ln -s "/content/data" "/content/drive/MyDrive/CS-GY6953/Proj1/proj1(augmix)"

### Build Dataset and wrap with DataLoader

In [ ]:
class CIFAR10CustomDataset(Dataset):
    def __init__(self, csv_file=None, img_dir=None, transform=None, is_test=False, dataframe=None):
        """Load images from a CSV or a preselected pandas DataFrame.

        Args:
            csv_file (str, optional): Path to CSV annotations.
            img_dir (str): Directory containing images.
            transform (callable, optional): Image preprocessing/augmentation.
            is_test (bool): For unlabeled test data, return IDs instead of labels.
            dataframe (pd.DataFrame, optional): Selected rows for a train/validation split.
                Supply exactly one of csv_file and dataframe.
        """
        # Reject both missing sources and two simultaneous sources to avoid ambiguous annotations.
        if (csv_file is None) == (dataframe is None):
            raise ValueError("Provide exactly one of csv_file or dataframe.")

        # Use CSV annotations or copy the selected split with a fresh positional index.
        self.df = (pd.read_csv(csv_file) if dataframe is None
                   else dataframe.reset_index(drop=True).copy())
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        # The number of annotation rows determines the number of available examples.
        return len(self.df)

    def __getitem__(self, index):
        # Look up the row by position and resolve its image filename.
        img_name = self.df.iloc[index]['filename']
        img_path = os.path.join(self.img_dir, img_name)

        # Close the file after decoding; RGB ensures three input channels for the network.
        with Image.open(img_path) as img:
            image = img.convert("RGB")

        # Apply the transform chosen for this dataset (random augmentation).
        if self.transform:
            image = self.transform(image)

        # Unlabeled test examples return IDs so predictions can be matched to submission rows.
        if self.is_test:
            img_id = self.df.iloc[index]['id']
            return image, img_id
        else:
            # Cross-entropy expects an integer class index stored as a long tensor.
            label = int(self.df.iloc[index]['label'])
            return image, torch.tensor(label, dtype=torch.long)

In [ ]:
# --- 1. Transforms & Augmentations ---
train_transforms = transforms.Compose([
    transforms.RandomCrop(32, padding=4),                   # Apply augmentation only to training images
    transforms.RandomHorizontalFlip(),                      # Randomly mirror training images to expose the model to additional variations.
    
    transforms.AugMix(                                      # Generate multiple augmentation chains and mix their results.
        severity=3,                                             # Strength of the individual transformations
        mixture_width=3,                                        # Number of independently augmented image chains
        chain_depth=-1,                                         # Randomly select 1–3 transformations per chain
        alpha=1.0                                               # Controls the randomness of the mixing weights
    ),
    
    transforms.ToTensor(),                                  # Convert RGB pixels to a C x H x W floating-point tensor scaled to [0, 1].
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.2023, 0.1994, 0.2010]),     # Standardize each channel using these fixed CIFAR-10 mean and standard-deviation values.
])

# Deterministic preprocessing for validation and Kaggle test images.
test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.2023, 0.1994, 0.2010])
])

# --- 2. Stratified 90/10 Training/Validation Split ---
labels_df = pd.read_csv('./data/train_labels.csv')
train_df, val_df = train_test_split(
    labels_df,
    test_size=0.1,              # Reserve 10% of labeled examples for validation; the remaining 90% are used for training.
    random_state=42,            # Fix split randomness so repeated runs use the same train/validation rows.
    stratify=labels_df['label'] # training and validation set get the same proportions of each class
)

train_dataset = CIFAR10CustomDataset(
    dataframe=train_df,
    img_dir='./data/train_images',
    transform=train_transforms,
    is_test=False
)

# Validation uses held-out labeled images WITHOUT random augmentation.
val_dataset = CIFAR10CustomDataset(
    dataframe=val_df,
    img_dir='./data/train_images',
    transform=test_transforms,
    is_test=False
)

# The separate Kaggle test set supplies IDs rather than ground-truth labels.
test_dataset = CIFAR10CustomDataset(
    csv_file='./data/test.csv',
    img_dir='./data/test_images',
    transform=test_transforms,
    is_test=True
)

# --- 3. Wrap in DataLoaders ---
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=128,
    shuffle=True,       # Change training example order on each pass through the loader.
    num_workers=4,      # Use four worker processes to load and transform examples in parallel.
    pin_memory=True     # Page-locked host tensors can speed up transfers to a CUDA device.
)
val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=128,
    shuffle=False,      # Keep validation/test iteration order stable.
    num_workers=4,
    pin_memory=True
)
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

print(f'Training: {len(train_dataset):,} | Validation: {len(val_dataset):,} | '
      f'Kaggle test: {len(test_dataset):,}')

### Veify labels and display images from each class

In [ ]:
# Confirm that training and validation use the same ten class IDs.
expected_labels = set(range(10))

assert set(train_dataset.df["label"].unique()) == expected_labels
assert set(val_dataset.df["label"].unique()) == expected_labels

# Display the number of images per class.
class_counts = pd.DataFrame({
    "Training": train_dataset.df["label"].value_counts().sort_index(),
    "Validation": val_dataset.df["label"].value_counts().sort_index()
})

print(class_counts)

# Define the normalization statistics used by train_transforms.
mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
std = torch.tensor([0.2023, 0.1994, 0.2010]).view(3, 1, 1)

# Display five augmented images from each class.
fig, axes = plt.subplots(10, 5, figsize=(10, 18))

for label in range(10):

    # Select five examples from the current class.
    examples = train_dataset.df[
        train_dataset.df["label"] == label
    ].sample(n=5, random_state=42)

    for col, (index, row) in enumerate(examples.iterrows()):

        # __getitem__ applies RandomCrop, RandomHorizontalFlip,
        # ColorJitter, Normalize, and RandomErasing.
        image_tensor, target = train_dataset[index]

        # Reverse normalization to restore RGB values for display.
        image = image_tensor * std + mean

        # Restrict pixel values to [0, 1].
        image = image.clamp(0, 1)

        # Convert from PyTorch's (C, H, W) to matplotlib's (H, W, C).
        image = image.permute(1, 2, 0).numpy()

        axes[label, col].imshow(image)
        axes[label, col].set_title(
            f"Label {target.item()}, ID {row['id']}"
        )
        axes[label, col].axis("off")

plt.tight_layout()
plt.show()


## Scaled Down ResNet18 Model

In [ ]:
class BasicBlock(nn.Module):

    expansion = 1 # BasicBlock preserves the requested channel count rather than expanding it.

    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        # The first 3 x 3 convolution can reduce spatial size when stride is greater than one.
        self.conv1 = nn.Conv2d(
            in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)

        # Normalize feature channels; the preceding convolution does not need a separate bias.
        self.bn1 = nn.BatchNorm2d(planes)

        # A second padded convolution preserves spatial size and produces the residual features.
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3,
                               stride=1, padding=1, bias=False)

        self.bn2 = nn.BatchNorm2d(planes)

        # An empty Sequential acts as the identity shortcut when input and output shapes agree.
        self.shortcut = nn.Sequential()

        # Project the shortcut when downsampling or changing channels so residual addition is valid.
        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion*planes)
            )

    def forward(self, x):

        out = F.relu(self.bn1(self.conv1(x))) # Apply convolution, batch normalization, and a nonlinearity to the main branch.
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x) # Add the shortcut to retain an easier path for information and gradients.
        out = F.relu(out) # Apply the final activation after combining the two branches.
        return out


In [ ]:
class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super(ResNet, self).__init__()
        # Track the channel count passed into the next residual block.
        self.in_planes = 40

        # A small stride-1 stem retains the 32 x 32 resolution of CIFAR-10 images.
        self.conv1 = nn.Conv2d(3, 40, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(40)

        # Four residual stages use widths 40, 80, 160, and 320; stages 2-4 halve spatial size.
        self.layer1 = self._make_layer(block, 40, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 80, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 160, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 320, num_blocks[3], stride=2)

        # Map the pooled features to one raw class score (logit) per class.
        self.linear = nn.Linear(320*block.expansion, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        # Only the first block of a stage performs any requested downsampling.
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion # The next block receives the channel count produced by this block.

        # Register the blocks as an ordered module so their parameters are tracked.
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))

        # Pass features through the four residual stages in order.
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)

        # For 32 x 32 inputs, the final 4 x 4 feature maps are averaged to one value per channel.
        out = F.avg_pool2d(out, 4)

        # Flatten pooled features while preserving the batch dimension.
        out = out.view(out.size(0), -1)

        # Return logits directly; CrossEntropyLoss handles the log-softmax internally.
        out = self.linear(out)
        return out

In [ ]:
def ResNet18():
    # Use two basic blocks per stage for the ResNet-18 layout with the reduced widths above.
    return ResNet(BasicBlock, [2, 2, 2, 2])

Instantiate Model and check the number of parameters:

In [ ]:
# Create a fresh model with newly initialized weights before configuring training.
model = ResNet18()

In [ ]:
# Inspect the architecture using one synthetic CIFAR-sized RGB input.
stats = summary(
    model,
    input_size=(1, 3, 32, 32),  # Dimensions are batch size, channels, height, and width.
    device="cpu", # Run this inspection on the CPU before the later training-device setup.
    verbose=0
)

# Compare all registered parameters with the subset eligible for gradient updates.
print(f"Total parameters: {stats.total_params:,}")
print(f"Trainable parameters: {stats.trainable_params:,}")

## Training

### Device Setup:

In [ ]:
# Select GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move model parameters and buffers to the same device that will hold each batch.
model = model.to(device)

# Enable cuDNN optimization for fixed-size images
if device.type == "cuda":
    # Let cuDNN select fast convolution algorithms for the fixed image shapes; results may be nondeterministic.
    cudnn.benchmark = True

print("Using device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


### Configure Training:

In [ ]:
# Training configuration
num_epochs = 300
learning_rate = 0.1

# Compare raw logits with integer class labels; the default reduction is the batch mean.
criterion = nn.CrossEntropyLoss()

# Update all model parameters with stochastic gradient descent.
optimizer = optim.SGD(
    model.parameters(),
    lr=learning_rate,
    momentum=0.9,       # Carry a running update direction to smooth and accelerate optimization.
    weight_decay=5e-4   # Apply L2-style regularization through SGD to discourage large weights.
)

# Reduce the learning rate along a cosine curve as training progresses.
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=num_epochs    # The loop steps this scheduler once per epoch, spanning the configured training run.
)

# Directory for saved model weights
checkpoint_dir = Path("checkpoints")

# Create the output directory and missing parents; reuse it if already present.
checkpoint_dir.mkdir(parents=True, exist_ok=True)


### Define Training and Validation Functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    # Enable training behavior: BatchNorm updates running statistics and dropout, if present, is active.
    model.train()

    # Reset sample-weighted loss and accuracy counters for this epoch only.
    running_loss = 0.0
    correct = 0
    total = 0

    # Visit every batch yielded by the training loader once during this epoch.
    for inputs, targets in loader:
        # Move both images and labels to the device holding the model parameters.
        inputs = inputs.to(device)
        targets = targets.to(device)

        # Clear previous gradients because backward() accumulates into parameter gradients.
        optimizer.zero_grad()

        # Forward pass: produce a matrix of logits with shape [batch size, number of classes].
        outputs = model(inputs)

        # Compute mean cross-entropy for this batch using logits and integer class indices.
        loss = criterion(outputs, targets)

        # Backpropagate through the recorded operations to compute parameter gradients.
        loss.backward()

        # Use those gradients, momentum, and weight decay to update the model parameters.
        optimizer.step()

        # Read the actual batch size, since the final batch may contain fewer examples.
        batch_size = targets.size(0)

        # Convert the batch mean to a summed loss; item() extracts a scalar without retaining its graph.
        running_loss += loss.item() * batch_size

        # Choose the largest logit per example; softmax would give the same winning class.
        predicted = outputs.argmax(dim=1)

        # Count correct predictions from this forward pass, before the parameter update is reflected in new outputs.
        correct += predicted.eq(targets).sum().item()

        # Count examples so loss and accuracy are weighted by samples rather than batches.
        total += batch_size

    # Divide summed losses by all processed examples (the loader must be nonempty).
    epoch_loss = running_loss / total

    # Report classification accuracy as a percentage, not a fraction.
    epoch_accuracy = 100.0 * correct / total

    # Return Python scalars for history tracking and the epoch progress display.
    return epoch_loss, epoch_accuracy


In [ ]:
# Disable gradient recording for the whole validation call to reduce memory use and computation.
@torch.no_grad()
def validate(model, loader, criterion, device):
    # Use stored BatchNorm statistics and disable dropout, if present; this is separate from no_grad().
    model.eval()

    # Reset counters so validation metrics describe only this pass over held-out examples.
    running_loss = 0.0
    correct = 0
    total = 0

    # Evaluate every validation batch without backward() or optimizer.step(), leaving learned parameters unchanged.
    for inputs, targets in loader:
        # Place images and labels on the same device as the model.
        inputs = inputs.to(device)
        targets = targets.to(device)

        # Compute class logits using the current learned parameters in evaluation mode.
        outputs = model(inputs)
        # Use the same loss definition as training so the two curves are comparable.
        loss = criterion(outputs, targets)

        # Use the actual number of examples, including any smaller final batch.
        batch_size = targets.size(0)
        # Accumulate the summed batch loss for a sample-weighted epoch mean.
        running_loss += loss.item() * batch_size

        # Select the highest-scoring class for each validation example.
        predicted = outputs.argmax(dim=1)
        # Count matching class indices and extract the scalar count.
        correct += predicted.eq(targets).sum().item()
        # Track the denominator shared by the final loss and accuracy calculations.
        total += batch_size

    # Average over examples, not batch means; the validation loader must be nonempty.
    epoch_loss = running_loss / total
    # Convert the fraction of correct predictions to percentage accuracy.
    epoch_accuracy = 100.0 * correct / total

    # Provide validation metrics for checkpoint selection, history, and logging.
    return epoch_loss, epoch_accuracy

### Train Model:

In [ ]:
# Store one value per completed epoch in each list for plotting or CSV analysis.
history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

# Track improvements during this run; this variable does not reload a previous checkpoint.
best_val_accuracy = 0.0

# Progress bar across all epochs
# The bar counts epochs; each iteration includes both training and validation.
progress_bar = tqdm(
    range(num_epochs),
    desc="Training ResNet-18",
    unit="epoch"
)

for epoch in progress_bar:
    # Measure the full epoch, including validation and saving results.
    start_time = time.time()

    # Train
    # This call re-enables training mode after the preceding epoch ended in evaluation mode.
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )

    # Validate
    # Measure held-out performance after all training updates for this epoch.
    val_loss, val_acc = validate(
        model, val_loader, criterion, device
    )

    # Record metrics
    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_acc)

    # Save best model
    # Overwrite the best weights only for a strict accuracy improvement; ties keep the earlier model.
    if val_acc > best_val_accuracy:
        best_val_accuracy = val_acc
        torch.save(
            # Save parameters and buffers only; optimizer and scheduler states are not included.
            model.state_dict(),
            checkpoint_dir / "best_model.pth"
        )

    # Update learning rate
    # Advance the schedule after training so the new learning rate is used in the next epoch.
    scheduler.step()

    # Save training history after every epoch
    # Rewrite the accumulated history after each epoch so completed metrics are saved independently.
    pd.DataFrame(history).to_csv(
        checkpoint_dir / "training_history.csv",
        index=False
    )

    elapsed = time.time() - start_time

    # Display results on the progress bar
    # Display metrics for the just-completed epoch while tqdm tracks progress through the total.
    progress_bar.set_postfix({
        "train_loss": f"{train_loss:.4f}",
        "val_loss": f"{val_loss:.4f}",
        "train_acc": f"{train_acc:.2f}%",
        "val_acc": f"{val_acc:.2f}%"
    })

    # Write an epoch summary to the SLURM log
    # Print a persistent summary without disrupting the progress-bar display.
    tqdm.write(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Acc: {val_acc:.2f}% | "
        f"Time: {elapsed:.1f}s"
    )

# Report the best score; model still holds the final epoch weights, not automatically the best saved weights.
print(f"\nBest validation accuracy: {best_val_accuracy:.2f}%")


## Plot Loss & Accuracy Curves

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, history["train_loss"], label="Training loss")
plt.plot(epochs, history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("ResNet-18 Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs, history["train_accuracy"], label="Training accuracy")
plt.plot(epochs, history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("ResNet-18 Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()


## Predict Labels of Test Set

Load the model with the best validation checkpoint

In [ ]:

# ============================================================
# Generate predictions for the unlabeled Kaggle test dataset
# ============================================================

# 1. Load the model weights with the highest validation accuracy.
# These may differ from the weights after the final training epoch.
model.load_state_dict(
    torch.load(
        checkpoint_dir / "best_model.pth",
        map_location=device,
        weights_only=True
    )
)

# Disable training-specific behavior, particularly BatchNorm updates.
model.eval()

# Store the image IDs and corresponding predicted class labels.
test_ids = []
test_predictions = []

# 2. Predict labels for every image in the Kaggle test dataset.
# No gradients are needed because we are only performing inference.
with torch.no_grad():
    for images, image_ids in tqdm(test_loader, desc="Predicting test images"):
        images = images.to(device)

        # Forward pass: shape (batch_size, 10), containing class logits.
        outputs = model(images)

        # Select the class with the highest logit for each image.
        predicted = outputs.argmax(dim=1)

        # Convert tensors to ordinary Python lists.
        test_ids.extend(
            image_ids.tolist() if torch.is_tensor(image_ids)
            else list(image_ids)
        )
        test_predictions.extend(predicted.cpu().tolist())

# 3. Associate each predicted label with its original image ID.
predictions_df = pd.DataFrame({
    "id": test_ids,
    "label": test_predictions
})

# Verify that every test image has exactly one prediction.
assert len(predictions_df) == len(test_dataset)
assert predictions_df["id"].is_unique

# 4. Read Kaggle's sample submission to preserve its row order.
sample_submission = pd.read_csv("./data/sample_submission.csv")

# Align predictions by image ID, rather than relying on row order.
submission = sample_submission[["id"]].merge(
    predictions_df,
    on="id",
    how="left",
    validate="one_to_one",
    sort=False
)

# 5. Validate the final submission format.
assert submission.columns.tolist() == sample_submission.columns.tolist()
assert len(submission) == len(sample_submission)
assert submission["label"].notna().all()
assert submission["label"].between(0, 9).all()

# 6. Save the submission CSV without the pandas row index.
submission.to_csv("submission.csv", index=False)

print(f"Saved submission.csv with {len(submission):,} predictions.")
print(submission.head())


## Visually inspect Kaggle test predictions

In [ ]:
submission = pd.read_csv("submission.csv")
test_df = pd.read_csv("./data/test.csv")

# Join predicted labels with their corresponding image filenames.
inspection_df = test_df.merge(
    submission,
    on="id",
    how="left",
    validate="one_to_one"
)

assert inspection_df["label"].notna().all()

# Randomly select 20 images for manual inspection.
examples = inspection_df.sample(n=20, random_state=42)

fig, axes = plt.subplots(4, 5, figsize=(12, 10))

for ax, (_, row) in zip(axes.flat, examples.iterrows()):
    image_path = Path("./data/test_images") / row["filename"]

    with Image.open(image_path) as img:
        ax.imshow(img.convert("RGB"))

    ax.set_title(
        f"ID: {row['id']}\nPredicted label: {row['label']}"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()
